
# Airbnb Price Analytics — ETL Pipeline

**Region:** United Kingdom  
**Cities:** London and Bristol  
**Currency:** GBP

This notebook processes Airbnb listings using the Bronze and Silver layers of the Medallion Architecture.

Bronze stores the original files, while Silver cleans the data and checks its quality.

The same pipeline works for different cities using the `city` parameter. Greater Manchester will be processed later to test how well our pipeline works on new data.



## 1. Configuration

We use the `city` parameter to select which city's data to process.

The source manifest (`sources.json`) contains the snapshot dates, file paths, and checksums. This allows us to use the correct files without manually changing the code.

Writing to Delta tables is disabled by default during testing.


In [ ]:
import json, re, hashlib, shutil, os, uuid
from pathlib import Path
from datetime import date, datetime, timezone
from pyspark.sql import functions as F
from pyspark.sql.window import Window
dbutils.widgets.text('city', 'london')
dbutils.widgets.text('catalog', 'workspace')
dbutils.widgets.text('source_root', '/Volumes/workspace/airbnb_eda/raw')
dbutils.widgets.text('bronze_root', '/Volumes/workspace/airbnb_bronze/raw')
dbutils.widgets.text('pipeline_version', 'etl-v1')
dbutils.widgets.dropdown('write_tables', 'false', ['false', 'true'])
silver_schema = "airbnb_silver"
gold_schema = "airbnb_gold"

In [ ]:
city = dbutils.widgets.get('city').strip().lower()
catalog = dbutils.widgets.get('catalog').strip()
source_root = dbutils.widgets.get('source_root').rstrip('/')
bronze_root = dbutils.widgets.get('bronze_root').rstrip('/')
pipeline_version = dbutils.widgets.get('pipeline_version').strip()
write_tables = dbutils.widgets.get('write_tables') == 'true'
if not re.fullmatch('[A-Za-z][A-Za-z0-9_]*', catalog):
    raise ValueError('Invalid catalog identifier')
if not pipeline_version:
    raise ValueError('pipeline_version cannot be empty')

In [ ]:
if not source_root.startswith(f'/Volumes/{catalog}/') or not bronze_root.startswith(f'/Volumes/{catalog}/'):
    raise ValueError('Volume roots must belong to the selected catalog')
print(f'city={city}, catalog={catalog}, pipeline_version={pipeline_version}, write_tables={write_tables}')


## 2. Bronze Layer

The Bronze layer stores the original Airbnb files without changing their content.

We copy the files from the EDA Volume and verify them using SHA-256 checksums. If a valid file already exists, we keep it instead of copying it again.

This helps us preserve the original data and makes the pipeline reproducible.


In [ ]:
manifest_path = Path(source_root) / 'sources.json'
with manifest_path.open(encoding='utf-8') as f:
    manifest = json.load(f)
sources = manifest['sources']
source_map = {s['city']: s for s in sources}
if len(sources) != len(source_map):
    raise ValueError('Duplicate city entries in manifest')
if city not in source_map:
    raise ValueError(f'Unsupported city {city!r}; available: {sorted(source_map)}')
s = source_map[city]
snapshot_date, source_url, expected_sha = (s['snapshot_date'], s['url'], s['sha256'].lower())

In [ ]:
if not re.fullmatch('\\d{4}-\\d{2}-\\d{2}', snapshot_date):
    raise ValueError('Invalid snapshot date')
date.fromisoformat(snapshot_date)
if not re.fullmatch('[0-9a-f]{64}', expected_sha):
    raise ValueError('Invalid SHA-256')
relative_path = f'data/raw/{city}/{snapshot_date}/listings.csv.gz'
expected_url = f'https://data.insideairbnb.com/united-kingdom/england/{city}/{snapshot_date}/data/listings.csv.gz'
if s['relative_path'] != relative_path or source_url != expected_url:
    raise ValueError('Unexpected source path / URL in pinned manifest')
source_path = Path(source_root) / relative_path
bronze_path = Path(bronze_root) / city / snapshot_date / 'listings.csv.gz'

In [ ]:
def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for part in iter(lambda: handle.read(8 * 1024 * 1024), b''):
            digest.update(part)
    return digest.hexdigest()
if not source_path.is_file() or sha256_file(source_path) != expected_sha:
    raise ValueError(f'Missing or corrupted EDA source: {source_path}')
bronze_path.parent.mkdir(parents=True, exist_ok=True)

In [ ]:

if bronze_path.exists():
    if sha256_file(bronze_path) != expected_sha:
        raise ValueError(
            f"Bronze already exists but checksum differs: {bronze_path}"
        )
    print("Existing Bronze verified")

else:
    try:
        with source_path.open("rb") as inp, bronze_path.open("xb") as out:
            shutil.copyfileobj(inp, out, length=8 * 1024 * 1024)
       
        if sha256_file(bronze_path) != expected_sha:
            raise ValueError("Bronze checksum mismatch after copying")

    except FileExistsError:
        if sha256_file(bronze_path) != expected_sha:
            raise ValueError("Existing Bronze file checksum mismatch")

    except Exception:
        bronze_path.unlink(missing_ok=True)
        raise

if sha256_file(bronze_path) != expected_sha:
    raise ValueError("Bronze validation failed")

print(f"Bronze verified: {city}, {snapshot_date}")


In [ ]:
if sha256_file(bronze_path) != expected_sha:
    raise ValueError('Bronze validation failed')
print(f'Bronze verified: {city}, {snapshot_date}, {bronze_path}, bytes={bronze_path.stat().st_size}')


## 3. Silver Layer

The Silver layer prepares the raw data for analysis and machine learning.

We convert columns into the correct data types, handle missing values, and identify data quality issues.

We keep suspicious records with quality flags instead of automatically removing them. Only the columns needed for further analysis are included.


In [ ]:
raw_df = (spark.read.option("header", True)
          .option("inferSchema", False).option("multiLine", True)
          .option("escape", '"').csv(str(bronze_path)))
required = ["id", "price", "room_type", "property_type", "accommodates", "bathrooms_text"]
missing_required = [c for c in required if c not in raw_df.columns]
if missing_required:
    raise ValueError(f"Missing required source columns: {missing_required}")

In [ ]:
# Only explicitly approved source fields can enter Silver. Optional absent fields become NULL.
source_fields = [
    "id", "price", "room_type", "property_type", "accommodates", "bedrooms", "beds",
    "bathrooms_text", "neighbourhood_cleansed", "latitude", "longitude",
    "minimum_nights", "maximum_nights", "instant_bookable", "last_scraped",
    "availability_30", "availability_60", "availability_90", "availability_365",
    "review_scores_rating"
]
missing_optional = [c for c in source_fields if c not in raw_df.columns and c not in required]
print("Missing optional source fields:", missing_optional)

In [ ]:
base = raw_df.select(*[
    (F.col(c) if c in raw_df.columns else F.lit(None).cast("string")).alias(c)
    for c in source_fields
])
input_rows = raw_df.count()
print("Input rows:", input_rows)


### 3.1 Price and Bathroom Cleaning

Prices are converted from text into numerical GBP values. The `$` symbol in the source files is only part of the export format, so no currency conversion is needed.

Missing or invalid prices are marked, but the records are kept.

We also extract the number of bathrooms and whether they are shared from `bathrooms_text`. Values that cannot be recognized remain missing.


In [ ]:
df = base.withColumn('price_raw', F.col('price'))
price_text = F.trim(F.col('price_raw'))
price_pattern = '^\\$?(?:[0-9]+|[1-9][0-9]{0,2}(?:,[0-9]{3})+)(?:\\.[0-9]{1,2})?$'
df = df.withColumn('price_parse_valid', F.coalesce(price_text.rlike(price_pattern), F.lit(False)))
df = df.withColumn('price_gbp', F.when(F.col('price_parse_valid'), F.regexp_replace(price_text, '[$,]', '').cast('double')).otherwise(F.lit(None).cast('double')))
df = df.withColumn('price_status', F.when(price_text.isNull() | (price_text == ''), 'missing').when(~F.col('price_parse_valid') | F.col('price_gbp').isNull(), 'malformed').when(F.col('price_gbp') <= 0, 'non_positive').otherwise('valid'))
df = df.withColumn('target_valid', F.col('price_status') == 'valid')
df = df.withColumn('price_parse_failed', F.col('price_status') == 'malformed')
df = df.withColumn('price_extreme', F.col('target_valid') & ((F.col('price_gbp') < 10) | (F.col('price_gbp') > 2000)))
df = df.withColumn('currency', F.lit('GBP'))
df = df.withColumn('bathrooms_source', F.col('bathrooms_text'))

In [ ]:
bath_text = F.lower(F.trim(F.col('bathrooms_source')))
bath_numeric = '^(?:[0-9]+(?:\\.[0-9]+)?)\\s+(?:(?:shared|private)\\s+)?baths?$'
bath_half = '^(?:(?:shared|private)\\s+)?half[ -]?bath(?:room)?s?$'
recognised = F.coalesce(bath_text.rlike(bath_numeric) | bath_text.rlike(bath_half), F.lit(False))
number = F.regexp_extract(bath_text, '^([0-9]+(?:\\.[0-9]+)?)', 1)
df = df.withColumn('bathrooms', F.when(bath_text.rlike(bath_half), F.lit(0.5)).when(bath_text.rlike(bath_numeric), number.cast('double')).otherwise(F.lit(None).cast('double')))
df = df.withColumn('bathrooms_shared', F.when(recognised, bath_text.rlike('(^|\\s)shared\\s')).otherwise(F.lit(None).cast('boolean')))
df = df.withColumn('bathrooms_parse_failed', bath_text.isNotNull() & (bath_text != '') & ~recognised)
df = df.withColumn('bathrooms_suspicious', F.coalesce((F.col('bathrooms') <= 0) | (F.col('bathrooms') > 20), F.lit(False)))
display(df.groupBy('bathrooms_source', 'bathrooms', 'bathrooms_shared').count().orderBy(F.desc('count')).limit(12))


### 3.2 Numerical Columns and Data Quality

We convert numerical and date columns into the correct types.

We check for invalid values in guest capacity, bedrooms, beds, coordinates, availability, and review scores.

Missing values are kept as `NULL`. Invalid or unusual values receive quality flags rather than being automatically removed.


In [ ]:
num_types = {'accommodates': 'int', 'bedrooms': 'double', 'beds': 'double', 'minimum_nights': 'int', 'maximum_nights': 'int', 'latitude': 'double', 'longitude': 'double', 'availability_30': 'int', 'availability_60': 'int', 'availability_90': 'int', 'availability_365': 'int', 'review_scores_rating': 'double'}
for name, dtype in num_types.items():
    raw_txt = F.trim(F.col(name))
    parsed = F.expr(f'try_cast(`{name}` AS {dtype})')
    df = df.withColumn(f'{name}_parse_failed', raw_txt.isNotNull() & (raw_txt != '') & parsed.isNull())
    if name == 'accommodates':
        df = df.withColumn('accommodates_fractional', F.coalesce(F.expr('try_cast(accommodates AS DOUBLE)').isNotNull() & (F.expr('try_cast(accommodates AS DOUBLE)') % 1 != 0), F.lit(False)))
    df = df.withColumn(name, parsed)
for name in ['room_type', 'property_type', 'neighbourhood_cleansed']:
    df = df.withColumn(name, F.when(F.trim(F.col(name)) == '', None).otherwise(F.trim(F.col(name))))
df = df.withColumn('invalid_accommodates', F.coalesce((F.col('accommodates') <= 0) | F.col('accommodates_fractional'), F.lit(False)))

In [ ]:
df = df.withColumn('invalid_bedrooms', F.coalesce(F.col('bedrooms') < 0, F.lit(False)))
df = df.withColumn('invalid_beds', F.coalesce(F.col('beds') < 0, F.lit(False)))
df = df.withColumn('invalid_nights', F.coalesce((F.col('minimum_nights') <= 0) | (F.col('maximum_nights') <= 0) | (F.col('minimum_nights') > F.col('maximum_nights')), F.lit(False)))
instant = F.lower(F.trim(F.col('instant_bookable')))
df = df.withColumn('instant_bookable_parse_failed', instant.isNotNull() & (instant != '') & ~instant.isin('t', 'f', 'true', 'false', '1', '0'))
df = df.withColumn('instant_bookable', F.when(instant.isin('t', 'true', '1'), True).when(instant.isin('f', 'false', '0'), False).otherwise(F.lit(None).cast('boolean')))
df = df.withColumn('last_scraped_parse_failed', F.col('last_scraped').isNotNull() & F.expr('try_cast(last_scraped AS DATE)').isNull())
df = df.withColumn('last_scraped', F.expr('try_cast(last_scraped AS DATE)'))
df = df.withColumn('invalid_coordinates', F.coalesce(~F.col('latitude').between(-90, 90) | ~F.col('longitude').between(-180, 180), F.lit(False)))
invalid_availability = F.lit(False)

In [ ]:
for name, days in [('availability_30', 30), ('availability_60', 60), ('availability_90', 90), ('availability_365', 365)]:
    invalid_availability = invalid_availability | F.coalesce(~F.col(name).between(0, days), F.lit(False))
for small, large in [('availability_30', 'availability_60'), ('availability_60', 'availability_90'), ('availability_90', 'availability_365')]:
    invalid_availability = invalid_availability | F.coalesce(F.col(small) > F.col(large), F.lit(False))
df = df.withColumn('invalid_availability', invalid_availability)
df = df.withColumn('invalid_review_score', F.coalesce(~F.col('review_scores_rating').between(0, 5), F.lit(False)))
flag_cols = ['price_parse_failed', 'price_extreme', 'bathrooms_parse_failed', 'bathrooms_suspicious', 'invalid_accommodates', 'invalid_bedrooms', 'invalid_beds', 'invalid_nights', 'invalid_coordinates', 'invalid_availability', 'invalid_review_score', 'instant_bookable_parse_failed', 'last_scraped_parse_failed']
flag_cols += [f'{name}_parse_failed' for name in num_types]
flag_cols += ['accommodates_fractional']
flag_exprs = [F.when(F.col('price_status') == 'missing', F.lit('price_missing'))]
flag_exprs += [F.when(F.col('price_status') == 'non_positive', F.lit('price_non_positive'))]

In [ ]:
flag_exprs += [F.when(F.col(name), F.lit(name)) for name in flag_cols]
df = df.withColumn('quality_flags', F.filter(F.array(*flag_exprs), lambda x: x.isNotNull()))
summary = df.agg(*[F.sum(F.when(F.col(f), 1).otherwise(0)).alias(f) for f in flag_cols])


### 3.3 Listing Keys and Quarantine

Each listing is identified by its city, listing ID, and snapshot date.

Records with missing or duplicate keys are placed in a separate quarantine dataset to avoid problems when joining tables.

We also save information about the source file and pipeline version.


In [ ]:
df = (df.withColumn("city", F.lit(city))
      .withColumn("listing_id", F.trim(F.col("id")))
      .withColumn("snapshot_date", F.to_date(F.lit(snapshot_date))))
key_window = Window.partitionBy("city", "listing_id", "snapshot_date")
df = df.withColumn("missing_key", F.col("listing_id").isNull() | (F.col("listing_id") == ""))
df = df.withColumn("duplicate_key", F.count(F.lit(1)).over(key_window) > 1)
df = df.withColumn("quarantine_reason", F.when(F.col("missing_key"), "missing_listing_id")
    .when(F.col("duplicate_key"), "duplicate_listing_key")
    .otherwise(F.lit(None).cast("string")))

In [ ]:
df = (df.withColumn("source_url", F.lit(source_url))
      .withColumn("source_sha256", F.lit(expected_sha))
      .withColumn("ingested_at", F.current_timestamp())
      .withColumn("pipeline_version", F.lit(pipeline_version)))

In [ ]:
silver_columns = ["city", "listing_id", "snapshot_date", "source_url", "source_sha256",
    "last_scraped", "ingested_at", "pipeline_version", "price_raw", "price_gbp",
    "currency", "target_valid", "price_status", "price_parse_valid", "price_parse_failed",
    "price_extreme", "room_type", "property_type", "accommodates", "bedrooms", "beds",
    "bathrooms", "bathrooms_shared", "bathrooms_source", "neighbourhood_cleansed",
    "latitude", "longitude", "minimum_nights", "maximum_nights", "instant_bookable",
    "availability_30", "availability_60", "availability_90", "availability_365",
    "review_scores_rating", "quality_flags", "quarantine_reason", "missing_key", "duplicate_key"]
silver_columns += [f for f in flag_cols if f not in silver_columns]
silver_all = df.select(*silver_columns)

In [ ]:
quarantine_df = silver_all.filter(F.col("quarantine_reason").isNotNull())
silver_df = silver_all.filter(F.col("quarantine_reason").isNull())
valid_rows = silver_df.count()
quarantined_rows = quarantine_df.count()
assert valid_rows + quarantined_rows == input_rows, "Row conservation failed"
print(f"City {city}: input={input_rows}, silver={valid_rows}, quarantine={quarantined_rows}")


## 4. Data Validation

We check the number of processed records, duplicate keys, missing prices, and data quality issues.

These results help us verify that the cleaning steps worked correctly and that no records were lost unexpectedly.


In [ ]:
assert silver_df.filter(F.col('listing_id').isNull() | (F.col('listing_id') == '')).count() == 0
assert silver_df.groupBy('city', 'listing_id', 'snapshot_date').count().filter('count > 1').count() == 0
print('Silver columns:', len(silver_df.columns))
print('Quarantine columns:', len(quarantine_df.columns))
summary_df = silver_df.agg(F.count('*').alias('silver_rows'), F.sum(F.when(F.col('target_valid'), 1).otherwise(0)).alias('valid_price_rows'), F.sum(F.when(~F.col('target_valid'), 1).otherwise(0)).alias('invalid_price_rows'))
display(summary_df)
display(df.groupBy('price_status').count().orderBy('price_status'))
display(summary)


## 5. Saving Silver Tables

After validation, the cleaned records can be saved to `workspace.airbnb_silver.listings`.

Records with invalid keys are stored separately in `workspace.airbnb_silver.listings_quarantine`.

The pipeline processes one city at a time. Running it again for the same city and snapshot updates only that part of the data, while preserving other cities.


In [ ]:

silver_table = f"{catalog}.airbnb_silver.listings"
quarantine_table = f"{catalog}.airbnb_silver.listings_quarantine"

predicate = (
    f"city = '{city}' "
    f"AND snapshot_date = DATE '{snapshot_date}'"
)

def write_city_snapshot(frame, table):
    if not spark.catalog.tableExists(table):
        (
            frame.write.format("delta")
            .mode("overwrite")
            .partitionBy("city", "snapshot_date")
            .saveAsTable(table)
        )
    else:
        (
            frame.write.format("delta")
            .mode("overwrite")
            .option("replaceWhere", predicate)
            .saveAsTable(table)
        )


In [ ]:
if write_tables:
    write_city_snapshot(silver_df, silver_table)
    write_city_snapshot(quarantine_df, quarantine_table)
    print('Written:', silver_table, quarantine_table)
else:
    print('DRY RUN: nothing written. Set widget write_tables=true only after review.')

In [ ]:
%sql
SELECT city, snapshot_date, COUNT(*) AS listings
FROM workspace.airbnb_silver.listings
GROUP BY city, snapshot_date
ORDER BY city;

In [ ]:
%sql
SELECT
    city,
    COUNT(*) AS total_listings,
    COUNT_IF(target_valid) AS valid_prices,
    COUNT_IF(NOT target_valid) AS invalid_prices
FROM workspace.airbnb_silver.listings
GROUP BY city
ORDER BY city;


## 6. Gold Layer

Gold contains listing-level model inputs and aggregated indicators for the dashboard. Only approved model features are selected; no ML preprocessing or train/test split is fitted here. The ML owner will manage splits, predictions and model evaluation.


### 6.1 Model Features
The feature table preserves listing keys and the price target alongside the initial safe predictors. Missing targets remain available for later inference. A previously published city snapshot is checked, not silently overwritten, so saved train/test assignments stay intact.

In [ ]:
# Select the current city's cleaned records; do not read every city.
feature_columns = [
    'city', 'listing_id', 'snapshot_date', 'price_gbp', 'target_valid',
    'room_type', 'property_type', 'accommodates', 'bedrooms', 'beds',
    'bathrooms', 'latitude', 'longitude', 'minimum_nights', 'maximum_nights'
]
features_df = silver_df.select(*feature_columns)
features_df = (features_df.withColumn('feature_version', F.lit('v1'))
                          .withColumn('split', F.lit(None).cast('string')))

In [ ]:
# The model owner sets the train/test split later.
assert features_df.count() == valid_rows
assert features_df.filter(F.col('split').isNotNull()).count() == 0
assert not any('price_quote' in c or 'estimated_revenue' in c
               or 'estimated_occupancy' in c or c == 'review_scores_value'
               for c in features_df.columns)
print(f'Candidate features for {city}: {valid_rows} listings')

In [ ]:
# Published features may already include ML-assigned splits.
gold_features_table = f'{catalog}.{gold_schema}.features'
key_filter = ((F.col('city') == city) &
              (F.col('snapshot_date') == F.to_date(F.lit(snapshot_date))))
feature_data_columns = [c for c in features_df.columns if c != 'split']

In [ ]:
def publish_features():
    if not spark.catalog.tableExists(gold_features_table):
        features_df.write.format('delta').partitionBy('city', 'snapshot_date').saveAsTable(gold_features_table)
        return 'Created Gold features table'
    existing = spark.table(gold_features_table).filter(key_filter)
    if existing.limit(1).count() == 0:
        write_city_snapshot(features_df, gold_features_table)
        return 'Added new city snapshot to Gold features'
    if set(existing.columns) != set(features_df.columns):
        raise ValueError('Existing feature schema differs; review versioning before changes')
    if (existing.select(*feature_data_columns).exceptAll(features_df.select(*feature_data_columns)).limit(1).count()
        or features_df.select(*feature_data_columns).exceptAll(existing.select(*feature_data_columns)).limit(1).count()):
        raise ValueError('Published features differ; do not overwrite ML data or splits')
    return 'Existing Gold features verified; split assignments preserved'

In [ ]:
if write_tables:
    print(publish_features())
else:
    print('DRY RUN: Gold features not written')

### 6.2 City Metrics
The city overview includes listing counts, valid price counts and median price, availability, review scores and distinct hosts. Each average has its own valid-row count; invalid availability and rating values are excluded from those averages.

In [ ]:
# Build only the current city's metrics, using Silver quality flags.
price_ok = F.when(F.col('target_valid'), F.col('price_gbp'))
avail_ok = F.when(~F.col('invalid_availability'), F.col('availability_365'))
review_ok = F.when(~F.col('invalid_review_score'), F.col('review_scores_rating'))

In [ ]:
city_metrics_df = silver_df.groupBy('city', 'snapshot_date').agg(
    F.count('*').alias('total_listings'),
    F.count('price_gbp').alias('observed_price_listings'),
    F.count(price_ok).alias('valid_price_listings'),
    F.percentile_approx(price_ok, 0.5).alias('median_price_gbp'),
    F.count(avail_ok).alias('valid_availability_listings'),
    F.avg(avail_ok).alias('avg_availability_365'),
    F.count(review_ok).alias('valid_review_listings'),
    F.avg(review_ok).alias('avg_review_score')
)

In [ ]:
# Count only hosts of listings that passed the key checks.
# Host IDs are used temporarily and are never published.
if 'host_id' in raw_df.columns:
    valid_keys = silver_df.select('listing_id').distinct()
    raw_hosts = raw_df.selectExpr('trim(id) AS listing_id', 'host_id')
    distinct_hosts = raw_hosts.join(valid_keys, 'listing_id', 'inner').where(
        F.col('host_id').isNotNull() & (F.trim('host_id') != '')
    ).select('host_id').distinct().count()
    city_metrics_df = city_metrics_df.withColumn('distinct_hosts', F.lit(distinct_hosts).cast('long'))
else:
    city_metrics_df = city_metrics_df.withColumn('distinct_hosts', F.lit(None).cast('long'))

In [ ]:
assert city_metrics_df.count() == 1
assert city_metrics_df.select('total_listings').first()[0] == valid_rows
display(city_metrics_df)

### 6.3 Room Type and Neighbourhood Metrics
Additional Gold summaries support room-type comparisons, neighbourhood price rankings and Genie questions. Rows without a valid target are counted as listings but excluded from median-price calculations.

In [ ]:
def segmented_metrics(dimension):
    return silver_df.groupBy('city', 'snapshot_date', dimension).agg(
        F.count('*').alias('total_listings'),
        F.count(F.when(F.col('target_valid'), 1)).alias('valid_price_listings'),
        F.percentile_approx(F.when(F.col('target_valid'), F.col('price_gbp')), 0.5).alias('median_price_gbp')
    )

room_type_metrics_df = segmented_metrics('room_type')
neighbourhood_metrics_df = segmented_metrics('neighbourhood_cleansed')

In [ ]:
assert room_type_metrics_df.agg(F.sum('total_listings')).first()[0] == valid_rows
assert neighbourhood_metrics_df.agg(F.sum('total_listings')).first()[0] == valid_rows
print('Room type groups:', room_type_metrics_df.count())
print('Neighbourhood groups:', neighbourhood_metrics_df.count())

### 6.4 Save Gold Metrics
Summary tables are updated by city and snapshot. Unlike the versioned feature table, these aggregates can be recomputed from the same Silver data without changing model splits.

In [ ]:
gold_metric_frames = {
    'city_metrics': city_metrics_df,
    'room_type_metrics': room_type_metrics_df,
    'neighbourhood_metrics': neighbourhood_metrics_df,
}

In [ ]:
if write_tables:
    for name, frame in gold_metric_frames.items():
        destination = f'{catalog}.{gold_schema}.{name}'
        write_city_snapshot(frame, destination)
        print('Written:', destination)
else:
    print('DRY RUN: Gold metric tables not written')

## 7. Final Validation
Checks confirm that the selected city and snapshot were written with the expected number of listings. The pipeline does not create predictions: they will be written by the ML owner after the train/test split and model evaluation.

In [ ]:
# Read-back checks are performed only after writes.
if write_tables:
    stored = spark.table(silver_table).filter(key_filter)
    stored_features = spark.table(gold_features_table).filter(key_filter)
    assert stored.count() == valid_rows
    assert stored_features.count() == valid_rows
    assert spark.table(f'{catalog}.{gold_schema}.city_metrics').filter(key_filter).count() == 1
    print(f'Validated persisted Silver and Gold for {city} ({snapshot_date})')
else:
    print(f'DRY RUN validated in memory for {city} ({snapshot_date})')

## 8. Release and City C
London and Bristol are processed with the same notebook and different city parameters. Before accessing Greater Manchester, the team will freeze the A/B pipeline, model, split assignments, baseline and test metrics, and source manifests. The first City C run must use those unchanged versions; any later fixes will be documented separately.

In [ ]:
%sql
SELECT 'features' AS table_name, city, COUNT(*) AS records
FROM workspace.airbnb_gold.features
GROUP BY city

UNION ALL

SELECT 'city_metrics', city, COUNT(*)
FROM workspace.airbnb_gold.city_metrics
GROUP BY city

UNION ALL

SELECT 'room_type_metrics', city, COUNT(*)
FROM workspace.airbnb_gold.room_type_metrics
GROUP BY city

UNION ALL

SELECT 'neighbourhood_metrics', city, COUNT(*)
FROM workspace.airbnb_gold.neighbourhood_metrics
GROUP BY city

ORDER BY table_name, city;